# 04 · Análisis

ANOVA de dos factores con réplicas, regresión del desempeño vs `r` (rendimientos decrecientes), gráficos de desempeño/VRAM/tiempo/costo y comparación de costo contra una API.

In [ ]:
# === Setup (en Colab se ejecuta solo; en local no hace nada) ===
import os, sys
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/triaje-clinico-lora"):
        !git clone https://github.com/Hammonne/triaje-clinico-lora.git /content/triaje-clinico-lora
    %cd /content/triaje-clinico-lora
    !git pull -q
    !pip install -q -r requirements.txt
    !pip install -q -e ".[gpu]"

In [ ]:
# === Celda de configuración (única fuente de parámetros) ===
import sys; sys.path.insert(0, "src")
from pathlib import Path
from triaje.config import Config, set_seed

config = Config()
# En Colab, las salidas van a Drive para sobrevivir desconexiones.
if EN_COLAB:
    config.drive_dir = Path("/content/drive/MyDrive/triaje")
    config.results_dir = config.drive_dir / "results"
    config.output_dir  = config.drive_dir / "adaptadores"
    config.results_dir.mkdir(parents=True, exist_ok=True)
set_seed(config.seed)
print("run_id ejemplo:", config.run_id(), "| modelo:", config.model_name, "| resultados en:", config.results_dir)

## Cargar resultados

In [ ]:
from triaje import analisis as A
df = A.cargar_resultados(config.results_dir / "resultados.csv")
df.head()

## ANOVA de dos factores (factores: r y alpha/r; réplicas: semillas)

In [ ]:
print("LoRA:\n", A.anova_dos_factores(df, "lora").round(4))
# print("DoRA:\n", A.anova_dos_factores(df, "dora").round(4))

## Regresión del desempeño vs r y rendimientos decrecientes

In [ ]:
reg = A.regresion_desempeno_vs_r(df, "lora")
print("R2 por modelo:", reg.r2)
print("Mejor modelo:", reg.mejor_modelo)
print("r de rendimientos decrecientes (aprox):", reg.r_rendimientos_decrecientes)

## Gráficos (desempeño, VRAM, tiempo, costo, sub-triaje)

In [ ]:
rutas = A.generar_todos_los_graficos(df, config.results_dir)
for r in rutas: print(r)

## Comparación de costo: API de pago vs. nuestro LoRA

In [ ]:
cmp = A.comparar_costo_api(df, n_test=len(df), tokens_in_prom=250, tokens_out_prom=120,
                           precio_in_1m=config.precio_api_usd_por_1m_tokens_in,
                           precio_out_1m=config.precio_api_usd_por_1m_tokens_out)
print(cmp)